In [2]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from lightgbm import LGBMRegressor

RANDOM_STATE = 42
CLEAN_CSV_PATH = "AirfoilDatset_cleaned_1.csv"

TARGET_CL = "coefficientLift"
TARGET_CD = "coefficientDrag"

# -----------------------------
# Helper metrics (works with older sklearn)
# -----------------------------
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def report_metrics(title, y_true, y_pred):
    print(f"\n{title}")
    print(f"  MAE : {mean_absolute_error(y_true, y_pred):.6f}")
    print(f"  RMSE: {rmse(y_true, y_pred):.6f}")
    print(f"  R²  : {r2_score(y_true, y_pred):.6f}")


In [3]:
# -----------------------------
# 1) Load data
# -----------------------------
print("\nLoading cleaned dataset...")
df = pd.read_csv(CLEAN_CSV_PATH, low_memory=False)
print("Shape:", df.shape)


Loading cleaned dataset...
Shape: (838207, 71)


In [4]:
 
# 2) Basic cleaning / type fixes
# -----------------------------
# Ensure XTR columns are numeric (prevents object-type errors)
for col in ["topXTR", "botXTR"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows that became NaN due to type conversion
cols_to_check = [c for c in ["topXTR", "botXTR"] if c in df.columns]
if cols_to_check:
    before = df.shape[0]
    df = df.dropna(subset=cols_to_check)
    after = df.shape[0]
    print(f"Dropped {before-after} rows due to non-numeric values in {cols_to_check}")

Dropped 1 rows due to non-numeric values in ['topXTR', 'botXTR']


In [5]:
# -----------------------------
# 3) Encode airfoilName (if exists) and save encoder
# -----------------------------
airfoil_le = None
if "airfoilName" in df.columns:
    airfoil_le = LabelEncoder()
    df["airfoilName_encoded"] = airfoil_le.fit_transform(df["airfoilName"].astype(str))
    print("Encoded 'airfoilName' -> 'airfoilName_encoded'")
else:
    print("No 'airfoilName' column found (OK if you already encoded it).")


Encoded 'airfoilName' -> 'airfoilName_encoded'


In [6]:
# -----------------------------
# 4) Build features/targets
# -----------------------------
assert TARGET_CL in df.columns, f"Missing target: {TARGET_CL}"
assert TARGET_CD in df.columns, f"Missing target: {TARGET_CD}"

# Drop raw airfoilName to keep only numeric
drop_cols = []
if "airfoilName" in df.columns:
    drop_cols.append("airfoilName")

X = df.drop(columns=[TARGET_CL, TARGET_CD] + drop_cols)
y_cl = df[TARGET_CL]
y_cd = df[TARGET_CD]

# Final safety: remove any remaining object columns
obj_cols = X.select_dtypes(include=["object"]).columns.tolist()
if obj_cols:
    # Try to coerce to numeric; if still object, drop them (rare)
    for c in obj_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    still_obj = X.select_dtypes(include=["object"]).columns.tolist()
    if still_obj:
        raise ValueError(f"Non-numeric columns still present in X: {still_obj}")
    # Drop rows that became NaN from coercion
    X = X.dropna()
    y_cl = y_cl.loc[X.index]
    y_cd = y_cd.loc[X.index]

feature_cols = X.columns.tolist()
print("Final feature count:", len(feature_cols))


Final feature count: 69


In [ ]:
# -----------------------------
# 5) Train/test split for deployment verification
# (You can show these metrics in 5.3 as evidence)
# -----------------------------
X_train, X_test, y_cl_train, y_cl_test = train_test_split(
    X, y_cl, test_size=0.20, random_state=RANDOM_STATE
)
y_cd_train = y_cd.loc[y_cl_train.index]
y_cd_test  = y_cd.loc[y_cl_test.index]

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


Train shape: (670564, 69) Test shape: (167642, 69)


In [ ]:
# -----------------------------
# 6) FINAL MODELS (match your Chapter 5 selection)
# -----------------------------
# Final Cl model: Random Forest
final_rf_cl = RandomForestRegressor(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=2
)

# Final Cd model: Tuned LightGBM
# IMPORTANT: If you have best_params_ from your tuning, paste them here.
final_lgbm_cd = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE
)


In [1]:
# -----------------------------
# 7) Fit models (for verification on test set)
# -----------------------------
print("\nTraining final models...")
final_rf_cl.fit(X_train, y_cl_train)
final_lgbm_cd.fit(X_train, y_cd_train)
print("Training completed.")

# Verification metrics on held-out test set (deployment evidence)
pred_cl_test = final_rf_cl.predict(X_test)
pred_cd_test = final_lgbm_cd.predict(X_test)

report_metrics("Deployment Verification (Random Forest -> Cl) [Test Set]", y_cl_test, pred_cl_test)
report_metrics("Deployment Verification (Tuned LightGBM -> Cd) [Test Set]", y_cd_test, pred_cd_test)



Training final models...


NameError: name 'final_rf_cl' is not defined

In [ ]:
# -----------------------------
# 8) Refit on FULL dataset (optional but recommended for final saved model)
# This matches “trained on full dataset” concept
# -----------------------------
print("\nRefitting models on FULL dataset for final deployment artifacts...")
final_rf_cl.fit(X, y_cl)
final_lgbm_cd.fit(X, y_cd)
print("Refit completed.")


In [ ]:
# -----------------------------
# 9) Save deployment artifacts
# -----------------------------
joblib.dump(final_rf_cl, "final_rf_cl_model.pkl")
joblib.dump(final_lgbm_cd, "final_lgbm_cd_model.pkl")
joblib.dump(feature_cols, "feature_columns.pkl")

if airfoil_le is not None:
    joblib.dump(airfoil_le, "airfoil_label_encoder.pkl")

print("\nSaved files:")
print(" - final_rf_cl_model.pkl")
print(" - final_lgbm_cd_model.pkl")
print(" - feature_columns.pkl")
if airfoil_le is not None:
    print(" - airfoil_label_encoder.pkl")

In [ ]:
# -----------------------------
# 10) Load + Inference demo (TAKE SCREENSHOTS of this output)
# -----------------------------
print("\n--- Deployment Demo: Load models and predict on 5 unseen samples ---")
loaded_rf = joblib.load("final_rf_cl_model.pkl")
loaded_lgbm = joblib.load("final_lgbm_cd_model.pkl")
loaded_features = joblib.load("feature_columns.pkl")

demo = df.sample(5, random_state=RANDOM_STATE).copy()
# If airfoilName exists, ensure encoded column exists (already handled above)
X_demo = demo.drop(columns=[TARGET_CL, TARGET_CD] + drop_cols, errors="ignore")[loaded_features]

pred_cl = loaded_rf.predict(X_demo)
pred_cd = loaded_lgbm.predict(X_demo)

demo_out = pd.DataFrame({
    "Actual_Cl": demo[TARGET_CL].values,
    "Predicted_Cl": pred_cl,
    "Actual_Cd": demo[TARGET_CD].values,
    "Predicted_Cd": pred_cd
})

print("\nSample prediction output (use this for screenshots):")
print(demo_out)

print("\nAll done. Models deployed locally and inference verified.")